# 🚀 Phase 0: System Setup, Hardware Verification & Phase Control

This notebook orchestrates the entire system:
1. **Environment & Auth:** Sets the NGC key and logs Docker into the registry.
2. **Hardware Check:** Verifies GPU drivers & utilization.
3. **Phase Control:** Dynamically switches between **Phase 1 (Large SDG LLM)** and **Phase 2 (Small Mistral LLM + NeMo Customizer)**.

## 1. Check Environment & Import Libraries

In [ ]:
import subprocess
import torch
import socket

print("=== 1. Hardware & GPU Check Inside Container ===")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# NVIDIA-SMI Integration
print("\n--- 1.1 NVIDIA-SMI GPU Status ---")
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(result.stdout)
except Exception as e:
    print(f"Could not execute nvidia-smi: {e}")

print("\n=== 2. Check Docker Network & Container Reachability ===")
# Services and their internal default ports in the Docker network
services = {
    "prometheus": 9090,
    "grafana": 3000,
    "dcgm-exporter": 9400
}

for service, port in services.items():
    try:
        # Attempt to establish a TCP connection to the service in the Docker network
        with socket.create_connection((service, port), timeout=2):
            print(f"[SUCCESS] Service '{service}' is reachable via port {port} in the Docker network.")
    except (socket.timeout, socket.error):
        print(f"[WARNING] Could not connect to '{service}' on port {port}.")

## 2. Environment & NGC Authentication

In [ ]:
import os
import subprocess

print("=== 3. NGC Authentication & Configuration ===")

api_key = os.getenv("NGC_API_KEY")

if api_key:
    # NGC stores the API key by default in this configuration file
    ngc_config_dir = os.path.expanduser("~/.ngc")
    os.makedirs(ngc_config_dir, exist_ok=True)
    
    config_file_path = os.path.join(ngc_config_dir, "config")
    
    # Write configuration file directly (avoids invoking the binary CLI)
    config_content = f"apikey = {api_key}\nformat = json\n"
    with open(config_file_path, "w") as f:
        f.write(config_content)
        
    print("[SUCCESS] NGC API key written directly to configuration.")
else:
    print("[INFO] No 'NGC_API_KEY' found. (Non-critical for local models such as your .nemo file).")
    print("[TIP] Set the key in the notebook via: os.environ['NGC_API_KEY'] = 'your_api_key'")

print("\n=== 4. Prepare Data Curation Directories ===")
raw_data_dir = "/data/nemo-fraud-detection/data/raw"
processed_data_dir = "/data/nemo-fraud-detection/data/sft"

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(processed_data_dir, exist_ok=True)

print(f"[INFO] Raw data folder ready: {raw_data_dir}")
print(f"[INFO] Processed folder ready: {processed_data_dir}")
print("[SUCCESS] Setup preparation completed successfully!")

---
## 3. Hardware & GPU Monitoring Check



In [ ]:
import logging
import subprocess
import torch

# Configure logging so outputs are displayed cleanly
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def run_hardware_check():
    logging.info("🔍 Starting hardware and GPU check...")

    if not torch.cuda.is_available():
        raise RuntimeError("❌ No GPU found! The system requires an NVIDIA L40S.")

    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    logging.info(f"✔ GPU detected: {gpu_name}")
    logging.info(f"✔ Available VRAM: {vram:.2f} GB")

    # Driver and monitoring check via nvidia-smi
    try:
        smi_result = subprocess.run(
            "nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader",
            shell=True, capture_output=True, text=True, check=True
        )
        logging.info(f"✔ Live metrics (nvidia-smi): {smi_result.stdout.strip()}")
    except Exception as e:
        logging.warning(f"⚠️ Could not retrieve nvidia-smi metrics: {e}")

# Run Hardware Check
run_hardware_check()
logging.info("🎉 System setup & verification completely finished!")

---
## 4. Prometheus and Grafana Setup

Replace the IP addresses with the public IP address from which Prometheus and Grafana are reachable.

In [ ]:
import logging
import requests

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def check_monitoring_stack():
    logging.info("=== 4. Monitoring Setup Check (Prometheus & Grafana) ===")
    
    monitoring_services = {
        "Prometheus": "http://18.153.96.36:9090/-/healthy",
        "Grafana": "http://18.153.96.36:3000/api/health"
    }
    
    for name, url in monitoring_services.items():
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                logging.info(f"✅ {name} is ready and responding (Status 200).")
            else:
                logging.warning(f"⚠️ {name} responded with status code: {response.status_code}")
        except Exception as e:
            logging.error(f"❌ {name} not reachable at {url}. Error: {e}")

    # Check whether DCGM-Exporter (which supplies GPU data to Prometheus) is running
    try:
        dcgm_metrics = requests.get("http://dcgm-exporter:9400/metrics", timeout=5)
        if "DCGM_FI_DEV_GPU_UTIL" in dcgm_metrics.text:
            logging.info("✅ DCGM-Exporter is providing GPU metrics!")
        else:
            logging.warning("⚠️ DCGM-Exporter is running, but is not providing GPU metrics.")
    except Exception:
        logging.error("❌ DCGM-Exporter not reachable on port 9400.")

check_monitoring_stack()

In [ ]:
import logging
import os
import wandb

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

def setup_weights_and_biases():
    logging.info("=== 6. Weights & Biases (W&B) Setup & Check ===")
    
    # Check whether the API key is set in environment variables
    wandb_api_key = os.getenv("WANDB_API_KEY")
    
    if wandb_api_key:
        logging.info("✔ WANDB_API_KEY found. Starting authentication...")
        try:
            # Attempt to log in
            wandb.login(key=wandb_api_key)
            logging.info("✅ Weights & Biases authenticated successfully!")
            
            # Short test run to verify the connection
            run = wandb.init(project="nemo-fraud-detection-setup", reinit=True, mode="online")
            logging.info(f"✅ W&B test run started (ID: {run.id})")
            wandb.finish()
            logging.info("✅ W&B test run finished successfully.")
            
        except Exception as e:
            logging.error(f"❌ Error during W&B authentication: {e}")
    else:
        logging.warning("⚠️ No 'WANDB_API_KEY' found.")
        logging.info("💡 Tip: Set the key in the notebook via: os.environ['WANDB_API_KEY'] = 'your_key'")
        logging.info("ℹ️ Training will still work without W&B (it will be skipped or logged locally only), but charts will be missing.")

# Execution
setup_weights_and_biases()

In [ ]:
import requests

try:
    response = requests.get("http://172.17.0.1:8800/v1/models", timeout=3)
    if response.status_code == 200:
        print("✅ NIM server is reachable!")
except Exception:
    print("❌ NIM server is not responding yet.")

In [ ]:
import os
import urllib.request
import json
from pathlib import Path
from dotenv import load_dotenv

def check_nemo_environment():
    print("=== 1. Load .env & Environment Variables ===")
    # Attempt to load .env (if present in current directory or under /data)
    load_dotenv()
    
    hf_token = os.getenv("HF_TOKEN")
    ngc_key = os.getenv("NGC_API_KEY")
    wandb_key = os.getenv("WANDB_API_KEY")
    
    print(f"🔑 HF_TOKEN present: {'✅ Yes' if hf_token else '❌ No'}")
    print(f"🔑 NGC_API_KEY present: {'✅ Yes' if ngc_key else '❌ No'}")
    print(f"🔑 WANDB_API_KEY present: {'✅ Yes' if wandb_key else '❌ No'}")
    
    print("\n=== 2. Directory Check (/data) ===")
    data_path = Path("/data")
    if data_path.exists():
        print(f"✅ Mount directory '/data' is reachable.")
        # Briefly display the contents of /data
        contents = [p.name for p in data_path.iterdir()]
        print(f"📁 Contents of /data: {contents}")
    else:
        print("⚠️ Warning: The directory '/data' was not found or is not mounted.")
        
    print("\n=== 3. Test Connection to LLM Container (nim-llama3) ===")
    # Inside the Docker network, we address the container by its service/container name:
    url = "http://nim-llama3:8000/v1/models"
    
    try:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            models = [m.get('id') for m in data.get('data', [])]
            print(f"🚀 Connection successful! Available models on the NIM server: {models}")
    except Exception as e:
        print(f"ℹ️ Could not reach the NIM server (nim-llama3) yet: {e}")
        print("   (Note: The LLM may still be loading in the background on initial startup.)")

if __name__ == "__main__":
    check_nemo_environment()